# LAMAR-m6A Tokenized Dataset 构建 Notebook

这个 notebook 对应旧代码中的 tokenizer 数据处理部分：

```python
AutoTokenizer.from_pretrained(...)
load_dataset("csv", ...)
dataset.map(...)
save_to_disk(...)
```

它会把 `train/valid/test` CSV 转成 HuggingFace `DatasetDict`。


## Step 0. 参数设置

默认读取：

```text
data/processed/full_splits/
data/processed/mature_splits/
```

默认输出：

```text
dataset/full/
dataset/mature/
```


In [5]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/picb/rnasys2/lidaoqian/LAMAR-M6A


In [1]:
import multiprocessing as mp

DATA_DIR = Path("data")
TOKENIZER_PATH = Path("tokenizer/single_nucleotide")
MODE = "both"  # full / mature / both
CELL_LINES = ["a549", "cd8t", "hek293", "hela", "molm13"]
LENGTHS = [51, 101, 201, 401, 801, 1601]
SPLITS = ["train", "valid", "test"]
NUM_PROC = mp.cpu_count()
OVERWRITE = False

MODE


'both'

## Step 1. 导入函数

对应文件：

```text
build_tokenized_dataset.py
```


In [6]:
from lamar_m6a.data.build_tokenized_dataset import (
    discover_split_csvs,
    mode_paths,
    build_one_dataset,
)

from transformers import AutoTokenizer
from datasets import load_dataset


/opt/anaconda3/envs/torch201/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Step 2. 发现 split CSV 文件

文件名需要符合：

```text
train_m6a_full_{cell}_{length}.csv
valid_m6a_full_{cell}_{length}.csv
test_m6a_full_{cell}_{length}.csv
```


In [ ]:
modes = ["full", "mature"] if MODE == "both" else [MODE]
all_groups = {}

for mode in modes:
    split_dir, output_dir = mode_paths(DATA_DIR, mode)
    groups = discover_split_csvs(split_dir, CELL_LINES, LENGTHS, SPLITS)
    all_groups[mode] = (split_dir, output_dir, groups)
    print("mode:", mode)
    print("split_dir:", split_dir)
    print("output_dir:", output_dir)
    print("groups:", len(groups))
    for key, files in list(sorted(groups.items()))[:3]:
        print(" ", key, sorted(files.keys()))


## Step 3. 构建 tokenized DatasetDict

每个 `{mode, cell, length}` 会保存成一个目录：

```text
dataset/full/m6a_full_{cell}_{length}/
dataset/mature/m6a_mature_{cell}_{length}/
```


In [ ]:
if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(f"Tokenizer path not found: {TOKENIZER_PATH}")

for mode, (split_dir, output_dir, groups) in all_groups.items():
    for (cell, length), files in sorted(groups.items()):
        missing = [split for split in SPLITS if split not in files]
        if missing:
            print(f"skip incomplete group: {mode} {cell} {length}; missing {missing}")
            continue

        dataset_name = f"m6a_{mode}_{cell}_{length}"
        output_path = output_dir / dataset_name
        if output_path.exists() and not OVERWRITE:
            print("skip existing:", output_path)
            continue

        print("processing:", mode, cell, length)
        build_one_dataset(
            data_files={split: files[split] for split in SPLITS},
            output_path=output_path,
            tokenizer_path=TOKENIZER_PATH,
            length=length,
            num_proc=NUM_PROC,
            overwrite=OVERWRITE,
        )


## Step 4. 检查输出

读取一个保存后的 HuggingFace Dataset，检查 split 和字段。


In [ ]:
from datasets import load_from_disk

for mode in modes:
    _, output_dir, groups = all_groups[mode]
    example = None
    for (cell, length), files in sorted(groups.items()):
        path = output_dir / f"m6a_{mode}_{cell}_{length}"
        if path.exists():
            example = path
            break
    if example is None:
        print(mode, "no saved dataset found")
        continue

    ds = load_from_disk(example)
    print("example:", example)
    print(ds)
    for split in ds.keys():
        print(split, ds[split].column_names, len(ds[split]))
